In [7]:
#in case we need this
%load_ext autoreload
%autoreload 2

import torch
import pandas as pd
import json
import torch.nn as nn
import numpy as np
import torch.optim as optim
from ft_model_def import FTTransformer  # WE are importing class here
from torch.utils.data import Dataset, DataLoader
from data_to_tensor import ACSDataset
from torch.utils.data import DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:

# set device for gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"found 1 gpu, using {device}")
else:
    print(f"found no gpu, using {device}")

found no gpu, using cpu


In [9]:
df_train = pd.read_csv('../3_Data_Preprocessing/preprocessing_data/baseline_train_engineered.csv')
df_test = pd.read_csv('../3_Data_Preprocessing/preprocessing_data/baseline_test_engineered.csv')

In [14]:
from collections import Counter
import json

# Load  metadata
with open('../3_Data_Preprocessing/preprocessing_data/feature_engineering_metadata.json') as f:
    metadata = json.load(f)

all_features = metadata['all_features']
categorical_features = metadata['categorical_features']
optimal_features = metadata['optimal_features']
label_encoder_classes = metadata['label_encoder_classes']

# Reconstruct label encoders from saved classes
label_encoders = {}
for col, classes in metadata['label_encoder_classes'].items():
    le = LabelEncoder()
    le.classes_ = np.array(classes)
    label_encoders[col] = le

# Prepare X, y
X_train = df_train[all_features].copy()
y_train = df_train['binary_target'].copy()
X_test = df_test[all_features].copy()
y_test = df_test['binary_target'].copy()

# Optimal feature subsets from dimensionality reduction
X_train_final = X_train[optimal_features].copy()
X_test_final = X_test[optimal_features].copy()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_train_final (optimal): {X_train_final.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'y_train distribution: {Counter(y_train)}')
print(f'y_test distribution:  {Counter(y_test)}')

X_train: (1469769, 26), y_train: (1469769,)
X_train_final (optimal): (1469769, 26)
X_test:  (304368, 26), y_test:  (304368,)
y_train distribution: Counter({0: 1114746, 1: 355023})
y_test distribution:  Counter({0: 233793, 1: 70575})


In [15]:


#  categorical
cat_cols = ['CIT', 'ENG', 'LANX', 'MAR', 'MIG', 'MSP', 'SEX',
                        'NATIVITY', 'ESR', 'OCCP', 'WKL', 'WRK', 'PUMA', 'CA_Region']


cont_cols = [
    'AGEP', 'WKHP', 'education_tier', 'race_ethnic_aggregate',
    'has_insurance', 'race_white', 'race_black',
    'race_asian', 'race_indigenous', 'race_other', 'is_latinx'
]


target_col = 'poverty_risk_score'

In [ ]:
#must handle nulls

#only need this if we are starting from raw data
for col in cat_cols:
    df_train[col] = df_train[col].astype(str)
    df_test[col]  = df_test[col].astype(str)

    le = label_encoders[col]

    df_train[col] = le.transform(df_train[col])

    df_test[col] = df_test[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

In [17]:
from sklearn.preprocessing import StandardScaler
#scalign down contfeatures

scaler = StandardScaler()

df_train[cont_cols] = scaler.fit_transform(df_train[cont_cols])
df_test[cont_cols]  = scaler.transform(df_test[cont_cols])

In [20]:
#must handle why cit is int?
df_train['CIT'] = df_train['CIT'].astype(int)
df_test['CIT']  = df_test['CIT'].astype(int)

In [19]:
for col in cat_cols + cont_cols:
    print(col, df_train[col].dtype)

CIT str
ENG int64
LANX int64
MAR int64
MIG int64
MSP int64
SEX int64
NATIVITY int64
ESR int64
OCCP int64
WKL int64
WRK int64
PUMA int64
CA_Region int64
AGEP float64
WKHP float64
education_tier float64
race_ethnic_aggregate float64
has_insurance float64
race_white float64
race_black float64
race_asian float64
race_indigenous float64
race_other float64
is_latinx float64


In [21]:
#build tensors
train_ds = ACSDataset(df_train, cat_cols, cont_cols, target_col)
test_ds  = ACSDataset(df_test,  cat_cols, cont_cols, target_col)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)



In [23]:
#fixing cardinality
cards = [
    int(df_train[col].max() + 1)
    for col in cat_cols
]

model = FTTransformer(
    cat_cardinalities=cards,
    n_cont=len(cont_cols),
    embed_dim=32,
    n_classes=4
).to(device)

criterion = torch.nn.CrossEntropyLoss()
#is this the best optimizer? keep reading
optimizer = optim.Adam(model.parameters(), lr=0.001)

/Users/ingridaltamirano/IdeaProjects/Capstone-Data-Ingestion-Test/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [24]:
import time
model.train()
for epoch in range(10):
    #o
    start_time = time.time()
    epoch_loss = 0
    for x_cat, x_cont, y in train_loader:
        x_cat, x_cont, y = x_cat.to(device), x_cont.to(device), y.to(device)

        optimizer.zero_grad()
        output = model(x_cat, x_cont)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_duration = time.time() - start_time

    print(f"Epoch {epoch+1} | "
          f"Avg Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Time: {epoch_duration:.2f}s")


KeyboardInterrupt: 

In [ ]:
#evaluation steps
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for x_cat, x_cont, y in test_loader:
        x_cat = x_cat.to(device)
        x_cont = x_cont.to(device)
        y = y.to(device)

        outputs = model(x_cat, x_cont)

        # Convert logits → predicted class
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

In [ ]:
print(classification_report(
    all_targets,
    all_preds,
    digits=4
))

In [ ]:
#confusion matrix
cm = confusion_matrix(all_targets, all_preds)
print(cm)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()